In [0]:
# 1) Configuración del bootstrap
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "credito_riesgo_lab"
TABLA_LANDING = f"{CATALOG}.{SCHEMA}.landing_clientes"   # simulación del sistema origen

N_CLIENTES = 10_000   # nº de clientes ficticios
SEED = 42             # semilla: mismos datos en cada ejecución (reproducible)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Esquema listo: {CATALOG}.{SCHEMA}")

In [0]:
# 2) Generar los clientes (simulamos el core bancario)
# spark.range(n) crea un DataFrame con una columna 'id' → la renombramos a cliente_id
clientes = spark.range(1, N_CLIENTES + 1).withColumnRenamed("id", "cliente_id")

# F.rand(semilla) devuelve un aleatorio entre 0 y 1; semilla distinta por variable
SECTORES = ["retail", "industria", "servicios", "agro", "construccion", "tecnologia"]

clientes_ficticios = (
    clientes
    .withColumn("r_ing",  F.rand(SEED + 1))
    .withColumn("r_dti",  F.rand(SEED + 2))
    .withColumn("r_ltv",  F.rand(SEED + 3))
    .withColumn("r_ant",  F.rand(SEED + 4))
    .withColumn("r_edad", F.rand(SEED + 5))
    .withColumn("r_pre",  F.rand(SEED + 6))
    .withColumn("r_sec",  F.rand(SEED + 7))
    .withColumn("r_pag",  F.rand(SEED + 8))
    .withColumn("ingresos",        F.round(600 + F.col("r_ing") * 5000, 2))    # € al mes
    .withColumn("endeudamiento",   F.round(0.05 + F.col("r_dti") * 0.75, 3))   # ratio deuda/ingresos (DTI)
    .withColumn("ltv",             F.round(0.10 + F.col("r_ltv") * 1.00, 3))   # loan-to-value
    .withColumn("antiguedad_meses", (F.col("r_ant") * 180).cast("int"))
    .withColumn("edad",            (18 + F.col("r_edad") * 47).cast("int"))
    .withColumn("num_prestamos",   (1 + F.col("r_pre") * 4).cast("int"))
    .withColumn("sector", F.element_at(
        F.array(*[F.lit(s) for s in SECTORES]),
        (F.col("r_sec") * 6 + 1).cast("int"),
    ))
    .withColumn("historial_pagos",
        F.when(F.col("r_pag") < 0.65, F.lit("puntual"))
         .when(F.col("r_pag") < 0.90, F.lit("atrasos_ocasionales"))
         .otherwise(F.lit("moroso")))
    .drop("r_ing", "r_dti", "r_ltv", "r_ant", "r_edad", "r_pre", "r_sec", "r_pag")
)

clientes_ficticios.printSchema()

In [0]:
# 3) "Suciedad" realista y escritura en LANDING
# En un banco real, la suciedad viene del sistema origen: reenvíos duplicados, campos vacíos...
# La simulamos: 2% duplicados + 1% ingresos nulos
df_landing = (
    clientes_ficticios
    .unionByName(clientes_ficticios.sample(fraction=0.02, seed=SEED + 99))
    .withColumn("ingresos",
        F.when(F.rand(SEED + 10) < 0.01, F.lit(None)).otherwise(F.col("ingresos")))
)

# La landing NO lleva _fecha_ingesta: ese metadato lo añade la capa bronze al ingestar
(
    df_landing.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_LANDING)
)

n = spark.table(TABLA_LANDING).count()
print(f"Tabla LANDING creada (simula el sistema origen): {TABLA_LANDING} → {n} filas")

In [0]:
# 4) Vista rápida de la landing (fíjate en duplicados y nulos: eso "vino" del origen)
display(spark.table(TABLA_LANDING).limit(5))

# 00 · Bootstrap — Datos sintéticos (se ejecuta UNA VEZ, a mano)

**Qué simula:** el sistema origen de datos (por ejemplo, el *core bancario* que guarda clientes y préstamos). En un banco real, esta capa no existe: los datos llegan por **ingestión** (Lakeflow Connect, Auto Loader…).

**Qué hace:** genera 10.000 clientes ficticios con variables de riesgo de crédito y los deja en una tabla *landing* (`landing_clientes`), con "suciedad" realista: ~2% duplicados y ~1% nulos.

> ⚠️ Este notebook **NO forma parte del job**: es el punto de partida de la demo. Ejecútalo manualmente una sola vez.

**Mapping con la industria:** `landing_clientes` ≈ lo que Lakeflow Connect deja en bronze al ingerir del sistema origen.